# Sprint 5 - Single-Stage YOLO26n Improvement

**Date:** 2026-05-12  
**Focus:** improve the one-stage weapon detector before continuing with more two-phase experiments.

This notebook is a report notebook: it uses artifacts already generated in `runs/` and `docs/sprint5_figures/`. It should not require retraining or running inference again before the professor meeting.

## Main conclusion

The high-resolution one-stage run is better than the previous YOLO26n baseline.

| model | checkpoint | best threshold | TP | FP | FN | precision | recall | F1 |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| YOLO26n baseline, `imgsz=640` | `runs/detect/runs/yolo26n_full/weights/best.pt` | 0.20 | 290 | 512 | 1223 | 0.3616 | 0.1917 | 0.2505 |
| YOLO26n high-res, `imgsz=960` | `runs/single_stage/yolo26n_img9604/weights/best.pt` | 0.10 | 310 | 308 | 1203 | 0.5016 | 0.2049 | 0.2909 |

The `img9604` run improves F1 by about **+0.0404**, increases true positives from **290 to 310**, and reduces false positives from **512 to 308** compared with the best threshold found for the baseline.

In [1]:
# Optional: reload the saved sweep tables if you want to inspect them interactively.
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
base_sweep = pd.read_csv(ROOT / 'runs/single_stage/evaluation/yolo26n_full_test_threshold_sweep.csv')
img960_sweep = pd.read_csv(ROOT / 'runs/single_stage/evaluation/yolo26n_img9604_test_threshold_sweep.csv')

base_sweep.assign(model='baseline_img640').tail(3), img960_sweep.assign(model='img9604').head()

(    threshold   tp   fp    fn  precision    recall        f1  \
 8         0.3  244  250  1269   0.493927  0.161269  0.243149   
 9         0.4  186  120  1327   0.607843  0.122935  0.204508   
 10        0.5  143   62  1370   0.697561  0.094514  0.166473   
 
     detections_per_image  prediction_candidates            model  
 8               0.479146                    494  baseline_img640  
 9               0.296799                    306  baseline_img640  
 10              0.198836                    205  baseline_img640  ,
    threshold   tp    fp    fn  precision    recall        f1  \
 0       0.01  417  2713  1096   0.133227  0.275611  0.179625   
 1       0.03  368  1149  1145   0.242584  0.243225  0.242904   
 2       0.05  351   702  1162   0.333333  0.231989  0.273578   
 3       0.07  329   487  1184   0.403186  0.217449  0.282525   
 4       0.10  310   308  1203   0.501618  0.204891  0.290943   
 
    detections_per_image  prediction_candidates    model  
 0            

## Why image size helped

The paper argues that real CCTV weapon detection is hard because weapons are small, distant, blurred, and often partially occluded. Increasing YOLO's training image size from `640` to `960` is a direct adaptation of that idea: it preserves more detail from small weapons during training and validation.

The improvement is not huge because the test set remains very difficult, but it is meaningful: the model found more true weapons with fewer false positives at its best threshold.

## Threshold sweep

The best threshold changed after retraining. The baseline had its best F1 around `0.20`; the high-resolution run had its best F1 around `0.10`.

![Threshold sweep comparison](docs/sprint5_figures/single_stage_threshold_sweep_comparison.png)

## Training behavior

Several `img960` attempts exist in `runs/single_stage/`. The run to report is `yolo26n_img9604`, because it completed much farther and reached the strongest validation metrics.

![Img960 training attempts](docs/sprint5_figures/single_stage_img960_training_attempts.png)

## Final run artifacts

These are the main artifacts from `runs/single_stage/yolo26n_img9604/`.

### Training curves

![Training curves](docs/sprint5_figures/img9604_results.png)

### Confusion matrix

![Confusion matrix](docs/sprint5_figures/img9604_confusion_matrix.png)

## Detection curves

![F1 curve](docs/sprint5_figures/img9604_BoxF1_curve.png)

![Precision-recall curve](docs/sprint5_figures/img9604_BoxPR_curve.png)

## Visual examples from the test run

Legend:

- green box: ground-truth weapon
- cyan box: true positive prediction
- orange box: false positive prediction
- red box: false negative ground-truth weapon

These figures were generated from the saved `yolo26n_img9604_test_lowconf_predictions.csv` file at threshold `0.10`. No new YOLO inference was run to create them.

### Successful detection

![Successful detection](docs/sprint5_figures/example_success_img9604.jpg)

### Partial success

This type of frame is useful for discussion: the model localizes some weapons, but still leaves missed boxes and/or extra detections.

![Partial success](docs/sprint5_figures/example_partial_img9604.jpg)

### False positive example

![False positive](docs/sprint5_figures/example_false_positive_img9604.jpg)

### Missed weapon example

This failure mode is still the main limitation. Even at higher resolution, heavily occluded or small weapons remain difficult.

![Missed weapon](docs/sprint5_figures/example_miss_img9604.jpg)

## Interpretation for the professor

The Sprint 5 single-stage experiment supports a clear narrative:

1. The two-phase pipeline is useful as an ablation, but its gating step loses recall.
2. The one-stage YOLO26n remains the strongest practical baseline.
3. Raising image size to `960` improves the one-stage detector on the real Cam5 test split.
4. The best operating point for the new model is lower confidence (`0.10`), because CCTV threat detection benefits from recovering more candidate weapons.
5. Results are still low in absolute recall, which matches the paper's conclusion that real CCTV weapon detection remains an open problem.

The next technical step, after the professor feedback, would be a crop-assisted one-stage ensemble: run YOLO on the full frame and also on person crops, then merge detections. That should be treated as an extension, not as a replacement for the improved single-stage baseline.

In [2]:
# Optional: reproduce only the report tables from saved CSV files.
# This cell does not train and does not run YOLO inference.
summary = pd.DataFrame([
    {'model': 'YOLO26n baseline img640', 'threshold': 0.20, 'tp': 290, 'fp': 512, 'fn': 1223, 'precision': 0.361596, 'recall': 0.191672, 'f1': 0.250540},
    {'model': 'YOLO26n img9604', 'threshold': 0.10, 'tp': 310, 'fp': 308, 'fn': 1203, 'precision': 0.501618, 'recall': 0.204891, 'f1': 0.290943},
])
summary

,model,threshold,tp,fp,fn,precision,recall,f1
0,YOLO26n baseline img640,0.2,290,512,1223,0.361596,0.191672,0.250540
1,YOLO26n img9604,0.1,310,308,1203,0.501618,0.204891,0.290943


## Two-phase person crop quality check

These figures inspect the person crops generated in Step 1 of the two-phase pipeline. They use the already saved crop files and metadata under `data/interim/two_phase/`, so no new training or inference is required.

The goal is to verify whether the crop step is cutting people correctly and whether the crop contains the hand/weapon area often enough for Stage 2 to work.

### Saved crop samples

Green crops are labeled `hold`; blue crops are labeled `no_hold`. The footer shows how many ground-truth weapon centers matched the detected person crop.

![Two-phase person crop samples](docs/sprint5_figures/two_phase_person_crop_samples.jpg)

### Crop context examples

Color legend: red = ground-truth weapon, cyan = detected person box, yellow = padded crop saved to disk.

![Crop context 1](docs/sprint5_figures/two_phase_crop_context_1.jpg)

![Crop context 2](docs/sprint5_figures/two_phase_crop_context_2.jpg)

![Crop context 3](docs/sprint5_figures/two_phase_crop_context_3.jpg)

### Stage 0 miss examples

These frames contain weapons, but the person-detection stage did not produce a crop whose person box contained the weapon center. In those cases, the two-phase pipeline cannot recover the weapon later.

![Stage 0 miss examples](docs/sprint5_figures/two_phase_stage0_miss_examples.jpg)

### Crop quality interpretation

The crops are often reasonable when the person detector covers the full body and the hand area. However, the examples also show why the staged pipeline is fragile: if the crop misses the hand/weapon region, or if the weapon center lies outside the detected person box, Stage 2 never sees a useful weapon crop.

This supports the current conclusion: person crops are promising as an auxiliary signal, but they should probably be used together with full-frame YOLO rather than as a hard replacement for the one-stage detector.

## One-stage vs two-phase visual comparison

The following panels compare the same Cam5 test frames using three views: ground truth, the final one-stage `img9604` model, and the available two-phase pipeline predictions.

Legend: green = ground truth, cyan = true positive, orange = false positive, red = false negative.

### Case 1 - One-stage wins

The one-stage model detects weapons that the two-phase pipeline misses.

![One-stage wins](docs/sprint5_figures/comparison_one_stage_wins.jpg)

### Case 2 - Both detect something

Both methods detect at least one weapon, but the one-stage detector still recovers more true positives in this example.

![Both detect](docs/sprint5_figures/comparison_both_detect.jpg)

### Case 3 - Two-phase-only detection

This is a useful nuance for the discussion: the two-phase pipeline can occasionally find a weapon missed by one-stage, even though it is weaker overall.

![Two-phase only](docs/sprint5_figures/comparison_two_phase_only.jpg)

### Case 4 - Both miss

This kind of frame shows the core limitation from the paper: small and occluded weapons remain difficult even after improving the one-stage model.

![Both miss](docs/sprint5_figures/comparison_both_miss.jpg)

### Case 5 - False positive comparison

Both approaches can still produce false positives, but the two-phase pipeline can add extra errors because it runs multiple stages and reprojects detections from crops.

![False positive comparison](docs/sprint5_figures/comparison_false_positive.jpg)

## Final Sprint 5 position

For this sprint, the strongest position is:

> The project should keep the one-stage YOLO26n detector as the main model. The high-resolution `img9604` run is the best current checkpoint, while the Sprint 4 two-phase pipeline should be presented as an ablation that exposed recall loss from staged filtering.

This is the cleanest argument because it is supported by the fixed Cam5 test split, the threshold sweep, and visual examples.

## What still needs improvement

The main limitation is still false negatives. Many missed weapons are small, partially occluded, or visually similar to dark objects near a person's hand. Lowering the confidence threshold recovers more detections, but also increases false positives.